# MSCS 634 Project Deliverable 1

## Data Collection, Cleaning, and Exploration

**Authors:** Asha Kilaru and Venkatappareddy Monukonda

This notebook uses a CMS healthcare dataset to explore claims, spending, and beneficiary patterns. The goal is to prepare the data for future modeling by performing cleaning, feature preparation, and exploratory data analysis.

In [ ]:
import json
import re
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load the dataset directly from the CMS data API
url = "https://data.cms.gov/data-api/v1/dataset/bf6a5b3b-31ee-4abb-b1ad-2607a1e7510a/data"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"})
with urllib.request.urlopen(req, timeout=120) as response:
    raw_data = json.load(response)

df = pd.DataFrame(raw_data)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nFirst 5 rows:")
display(df.head())

### Step 1: Dataset Selection and Motivation

This section explains why the CMS healthcare dataset was chosen for this project. The dataset contains more than 500 records and several categorical and numerical variables, making it suitable for data cleaning, EDA, and future predictive modeling.

### Step 2: Data Cleaning and Preparation

This chunk focuses on preparing the dataset for analysis. The main tasks are handling missing values, removing duplicates, correcting inconsistent values, and identifying noisy data such as extreme spending values.

In [ ]:
# Review data types and missing values before cleaning
print(df.dtypes)
print("\nMissing values before cleaning:")
print(df.isna().sum())

# Convert numeric columns to numeric type
numeric_cols = ["Tot_Benes", "Tot_Clms", "Tot_Spndng", "Avg_Spnd_Per_Bene", "Avg_Spnd_Per_Clm"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove duplicate rows
df = df.drop_duplicates()

# Drop rows that are completely empty
df = df.dropna(how="all")

# Fill missing values for categorical columns with a placeholder
categorical_fill = ["Brnd_Name", "Gnrc_Name", "HCPCS_Cd", "HCPCS_Desc", "Year"]
for col in categorical_fill:
    df[col] = df[col].fillna("Unknown")

# Fill missing numeric values with the median of the column
for col in numeric_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# Clean the Year column by extracting the year from values such as 2025 (Q1-Q4)
def extract_year(value):
    if pd.isna(value):
        return np.nan
    text = str(value)
    match = re.search(r"(\d{4})", text)
    return int(match.group(1)) if match else np.nan

df["Year_Value"] = df["Year"].apply(extract_year)
df["Year_Value"] = df["Year_Value"].fillna(df["Year_Value"].median())

# Identify possible outliers in total spending using the IQR rule
q1 = df["Tot_Spndng"].quantile(0.25)
q3 = df["Tot_Spndng"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_count = ((df["Tot_Spndng"] < lower_bound) | (df["Tot_Spndng"] > upper_bound)).sum()

print("\nMissing values after cleaning:")
print(df.isna().sum())
print("\nRows after cleaning:", df.shape[0])
print("Outlier rows for total spending:", outlier_count)

# Save a cleaned version for future use
df.to_csv("cms_healthcare_claims_cleaned.csv", index=False)
print("Cleaned data written to cms_healthcare_claims_cleaned.csv")

### Step 3: Exploratory Data Analysis

This chunk explores the distribution of key variables and investigates relationships between spending, claims, and year. Visualizations are used to identify patterns, skewness, and possible outliers that may influence future modeling choices.

In [ ]:
# Distribution of total spending
plt.figure(figsize=(10, 6))
sns.histplot(df["Tot_Spndng"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of Total Spending")
plt.xlabel("Total Spending")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Top 10 generic names by total spending
top_generic = df.groupby("Gnrc_Name")["Tot_Spndng"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_generic.values, y=top_generic.index, palette="viridis")
plt.title("Top 10 Generic Names by Total Spending")
plt.xlabel("Total Spending")
plt.ylabel("Generic Name")
plt.tight_layout()
plt.show()

# Relationship between claims and spending
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="Tot_Clms", y="Tot_Spndng", alpha=0.6, color="darkorange")
plt.title("Total Claims vs Total Spending")
plt.xlabel("Total Claims")
plt.ylabel("Total Spending")
plt.tight_layout()
plt.show()

# Yearly spending trend
yearly_spending = df.groupby("Year_Value")["Tot_Spndng"].sum().sort_index()
plt.figure(figsize=(10, 6))
sns.lineplot(x=yearly_spending.index, y=yearly_spending.values, marker="o", color="green")
plt.title("Total Spending by Year")
plt.xlabel("Year")
plt.ylabel("Total Spending")
plt.tight_layout()
plt.show()

### Step 4: Insights and Future Modeling Direction

This final section summarizes the main findings from the EDA. These insights will guide future modeling steps by helping identify the most relevant features, target variables, and potential issues such as skewness or outliers.